# How SparkRules Works - A Complete Tutorial

This tutorial explains the SparkRules architecture from the ground up:

1. **DRL Parsing** - how rule text becomes an AST
2. **Compilation** - how rules are classified and optimized
3. **Alpha Network** - how shared predicates reduce evaluation cost
4. **Execution** - how facts are scored against rules
5. **Governance** - how rules are versioned and promoted

```bash
pip install sparkrules
```

## 1. DRL Parsing

SparkRules uses a Drools-style DRL (Drools Rule Language) syntax.
The parser converts DRL text into an Abstract Syntax Tree (AST).

In [ ]:
from sparkrules.parser import parse, parse_rules, print_ast

drl = """
rule "high-value-order"
  salience 10
  reason_codes ["HV001"]
  when
    $order : Order( $order.amount > 1000 )
  then
    result.risk = "high";
    result.review = true;
end
"""

ast = parse(drl)
print(f"Rule name: {ast.name}")
print(f"Salience: {ast.salience}")
print(f"Reason codes: {ast.reason_codes}")
print(f"When patterns: {len(ast.when)}")
print(f"Then actions: {len(ast.then)}")
print(f"\nPretty-printed DRL:")
print(print_ast(ast))

## 2. Compilation and Classification

The V2 engine classifies each rule into one of three execution strategies:

| Strategy | When used | How it works |
|----------|-----------|-------------|
| **SQL_PUSHDOWN** | Simple comparisons + literal actions | Translated to Spark SQL for Catalyst optimization |
| **ALPHA_SHARED** | Column predicates, not fully SQL-translatable | Shared boolean columns in Spark |
| **PYTHON_FALLBACK** | Multi-fact, complex expressions | mapPartitions with compiled closures |

In [ ]:
from sparkrules.compiler.rulepack import RulePack
from sparkrules.compiler.translator import translate_predicate

multi_drl = """
rule "simple" salience 10
  when $t : T ( $t.x > 5 )
  then result.ok = true; end

rule "regex" salience 5
  when $t : T ( $t.name matches "^A.*" )
  then result.starts_with_a = true; end

rule "multi-fact"
  when $a : A ( true ) and $b : B ( true )
  then result.joined = true; end
"""

pack = RulePack.from_drl(multi_drl)
for r in pack.rules:
    sql = r.predicate_sql or "N/A"
    print(f"{r.name:>12s}: {r.strategy.name:<18s} SQL: {sql}")

## 3. Alpha Network - Shared Predicate Evaluation

When multiple rules share the same predicate (e.g. `$t.score > 600`),
the alpha network evaluates it **once** and shares the result.

This is the key optimization from the Rete algorithm.

In [ ]:
from sparkrules.compiler.alpha_network import AlphaNetwork

# 10 rules all checking $t.score > 600
shared_drl = "\n".join(
    f'rule "r{i}" when $t : T ( $t.score > 600 ) then result.v{i} = {i}; end' for i in range(10)
)
rules = parse_rules(shared_drl)
net = AlphaNetwork.from_rules(rules)

print(f"Total predicates across all rules: {net.total_predicates}")
print(f"Unique alpha nodes (actual evaluations): {net.unique_alphas}")
print(f"Sharing ratio: {net.sharing_ratio:.1f}x")
print(f"\nWithout alpha network: 10 evaluations per fact")
print(f"With alpha network: {net.unique_alphas} evaluation per fact")
print(f"Savings: {(1 - net.unique_alphas / net.total_predicates) * 100:.0f}%")

## 4. Execution - Scoring Facts

The `LocalRuleExecutor` uses compiled closures and the alpha network
for fast evaluation. Results include fired status, action outputs,
and reason codes.

In [ ]:
from sparkrules.executor.local_executor import LocalRuleExecutor

lending_drl = """
rule "decline" salience 100 reason_codes ["CR001"]
  when $a : App( $a.fico < 580 )
  then result.decision = "DECLINE"; end

rule "refer" salience 50 reason_codes ["CR002"]
  when $a : App( $a.fico < 660 )
  then result.decision = "REFER"; end

rule "approve" salience 10
  when $a : App( $a.fico >= 660 )
  then result.decision = "APPROVE"; end
"""

executor = LocalRuleExecutor.from_drl(lending_drl)

# Score a single applicant
result = executor.score({"a": {"fico": 550}})
print("Single fact scoring:")
print(f"  Fired any: {result.fired_any}")
print(f"  Decision: {result.merged_actions.get('decision')}")
for f in result.fires:
    if f.fired:
        print(f"  Rule: {f.rule_name} (salience={f.salience}) codes={f.reason_codes}")

# Batch scoring
print("\nBatch scoring:")
facts = [{"a": {"fico": s}} for s in [550, 640, 720, 580, 800]]
results = executor.apply(facts)
for i, r in enumerate(results):
    print(f"  FICO={facts[i]['a']['fico']:>3d} -> {r.merged_actions.get('decision', 'N/A')}")

## 5. Governance - Versioning and Promotion

SparkRules supports rule versioning with dev -> stage -> prod promotion.
Rules are stored in a metadata store with active/inactive status.

In [ ]:
from sparkrules.store import InMemoryRuleMetadataStore
from sparkrules.model.rule import Rule, RuleDefinition, RuleFormat, new_rule_id
from datetime import UTC, datetime

store = InMemoryRuleMetadataStore()

# Create a rule version
rule = Rule(
    rule_id=new_rule_id(),
    rule_handle="credit-check",
    version=0,  # auto-incremented
    rule_group="underwriting",
    salience=100,
    effective_from=datetime.now(UTC),
    effective_to=None,
    is_active=True,
    rule_definition=RuleDefinition(
        source='rule "credit-check" when $a : App( $a.fico < 600 ) then result.d = "decline"; end',
        format=RuleFormat.DRL,
    ),
    namespace="lending",
)

stored = store.insert(rule)
print(f"Created: {stored.rule_handle} v{stored.version} (active={stored.is_active})")

# List versions
versions = store.list_versions("credit-check")
print(f"Versions: {len(versions)}")

# Active set hash (for deterministic replay)
hash_val = store.active_set_version(datetime.now(UTC))
print(f"Active set hash: {hash_val[:16]}...")

## Architecture Summary

```
DRL Text
  |-- Parser --> AST (RuleAst)
  |-- Classifier --> Strategy (SQL_PUSHDOWN / ALPHA_SHARED / PYTHON_FALLBACK)
  |-- Translator --> Spark SQL (for SQL_PUSHDOWN)
  |-- Closure Compiler --> Python callables (for ALPHA_SHARED / PYTHON_FALLBACK)
  |-- Alpha Network --> Shared predicate evaluation
  |-- RulePack --> Structured, classified, serializable
  |
  |-- LocalRuleExecutor --> Python-native scoring (sub-ms latency)
  |-- SparkRuleExecutor --> Three-strategy Spark dispatch
  |-- PandasExecutor --> Vectorized pandas evaluation
```

For more details:
- [Architecture docs](https://sparkrules.readthedocs.io)
- [GitHub repo](https://github.com/vaquarkhan/sparkrules)
- [AGENTS.md](https://github.com/vaquarkhan/sparkrules/blob/main/AGENTS.md) for AI coding agents